# Pheno4D - REAL plant growth as a 4D OpenUSD twin (CPU only)

**What this is:** the first *real agricultural data* in the twin. Pheno4D
(University of Bonn / ETH Zurich, published PLoS ONE 2021) scanned the SAME
maize and tomato plants daily in sub-millimeter 3D. We take one maize plant
(Maize01, ~14 daily scans), and author every day as **USD time samples** on
one `UsdGeomPoints` prim - which is exactly the twin's 4D timesample
mechanic, previously shown with synthetic NDVI, now with a real scanned,
growing plant. It then composes into the farm twin layer stack headless.

**What this is NOT:** a field plot and not photos->3DGS. It is laser-
scanned greenhouse plants (point clouds with per-point organ labels).
The photos->3DGS path is `gs_scan.ipynb`; field-plot point clouds are the
post-hackathon BonnBeetClouds3D item (docs/04 SS11).

**Cost:** CPU only, ~15-25 min (4.4 GB download + parse + USD authoring).


In [ ]:
import os
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = os.getcwd()
BASE = "/tmp/p4d" if ON_KAGGLE else "."
os.makedirs(BASE, exist_ok=True)
print("env:", "Kaggle" if ON_KAGGLE else "local", "| WORK:", WORK, "| BASE:", BASE)
!pip install -q "usd-core==26.8"
import pxr
print("pxr ok")


## 1. Data: Pheno4D release (direct download, no registration)
4.4 GB zip; we extract ONLY `Maize01` (~14 daily scans). Units in the zip
are centimetres in the scanner frame - converted to metres below.


In [ ]:
os.chdir(BASE)
import subprocess, glob, re
if not os.path.isdir("Pheno4D/Maize01"):
    url = "https://www.ipb.uni-bonn.de/html/projects/Pheno4D/Pheno4D.zip"
    subprocess.run(["wget", "-q", url], check=True)                 # 4.4 GB
    subprocess.run(["unzip", "-q", "Pheno4D.zip", "Pheno4D/Maize01/*"], check=True)
    os.remove("Pheno4D.zip")                                          # free 4.4 GB
files = sorted(glob.glob("Pheno4D/Maize01/M01_*.txt"),
               key=lambda f: int(re.search(r"M01_(\d{4})", f).group(1)))
assert files, "no Maize01 scans found"
dates = [re.search(r"M01_(\d{4})", f).group(1) for f in files]
print(f"{len(files)} daily scans: {dates[0]} .. {dates[-1]}")


## 2. Author `pheno_plant.usda` - one prim, time-sampled per scan day
Colors from the per-point organ labels (annotated `_a` files): soil =
brown, stem = gold, leaves = greens. Unlabeled days get a height gradient.
Growth table printed from the data itself.


In [ ]:
import numpy as np, math, shutil, json
from pxr import Sdf, Usd, UsdGeom, Gf, Vt

MAX_PTS_PER_DAY = 60_000

def load_day(path):
    raw = open(path).read()
    ncol = len(raw.splitlines()[0].split())
    arr = np.fromstring(raw, sep=" ")          # C speed, whitespace-run safe
    assert arr.size % ncol == 0, f"ragged file {path}"
    return arr.reshape(-1, ncol)

days = [load_day(f) for f in files]
z_all = np.concatenate([d[:, 2] for d in days])
SCALE = 0.01 if (z_all.max() - z_all.min()) > 30 else 1.0
print("unit scale applied:", SCALE, "(scanner cm -> twin metres)")
del z_all

label_hists = {}
for f, d in zip(files, days):
    if d.shape[1] == 5:
        v, c = np.unique(d[:, 3].astype(int), return_counts=True)
        label_hists[os.path.basename(f)] = dict(zip(v.tolist(), c.tolist()))
print("annotated files:", sum(1 for d in days if d.shape[1] == 5), "of", len(days),
      "| label histogram (first annotated):",
      next(iter(label_hists.values())) if label_hists else "none")

def colors_for(d):
    if d.shape[1] == 5:
        sem = d[:, 3].astype(int)
        inst = d[:, 4].astype(int)
        cols = np.zeros((len(d), 3), np.float32)
        cols[sem == 0] = (0.42, 0.33, 0.22)                    # soil
        cols[sem == 1] = (0.79, 0.64, 0.15)                    # stem
        leaf = sem >= 2
        palette = np.array([(0.10, 0.45, 0.15), (0.16, 0.60, 0.22), (0.30, 0.72, 0.30)], np.float32)
        cols[leaf] = palette[np.abs(inst[leaf]) % 3]           # leaves: greens by instance
        # unknown-label fallback
        unk = ~np.isin(sem, [0, 1]) & ~leaf
        cols[unk] = (0.5, 0.5, 0.5)
        return cols
    h = (d[:, 2] - d[:, 2].min()) / max(1e-9, d[:, 2].ptp())
    return np.stack([0.1 + 0.3 * h, 0.35 + 0.4 * h, 0.12 + 0.15 * h], 1).astype(np.float32)

layer = Sdf.Layer.Find("pheno_plant.usda") or Sdf.Layer.CreateNew("pheno_plant.usda")
stage = Usd.Stage.Open(layer)
root = UsdGeom.Xform.Define(stage, "/PhenoPlant")
stage.SetDefaultPrim(root.GetPrim())
pts = UsdGeom.Points.Define(stage, "/PhenoPlant/PlantCloud")

stats = []
for i, (f, d, date) in enumerate(zip(files, days, dates)):
    stride = max(1, math.ceil(len(d) / MAX_PTS_PER_DAY))
    s = d[::stride]
    xyz = np.stack([s[:, 0], s[:, 1], s[:, 2]], 1) * SCALE
    xy_med = np.median(xyz[:, :2], 0); xyz[:, :2] -= xy_med     # plant near origin
    cols = colors_for(s)
    t = Usd.TimeCode(i)
    pts.GetPointsAttr().Set(Vt.Vec3fArray([Gf.Vec3f(*p) for p in xyz]), t)
    pv = UsdGeom.PrimvarsAPI(pts.GetPrim()).CreatePrimvar("displayColor", Sdf.ValueTypeNames.Color3fArray, UsdGeom.Tokens.vertex)
    pv.Set(Vt.Vec3fArray([Gf.Vec3f(*c) for c in cols]), t)
    wa = pts.GetPrim().CreateAttribute("widths", Sdf.ValueTypeNames.FloatArray)
    wa.Set(Vt.FloatArray([0.006] * len(xyz)), t)
    wa.SetMetadata("interpolation", UsdGeom.Tokens.vertex)
    lo, hi = xyz.min(0), xyz.max(0)
    pts.CreateExtentAttr(Vt.Vec3fArray([Gf.Vec3f(*lo), Gf.Vec3f(*hi)]), True).Set(
        Vt.Vec3fArray([Gf.Vec3f(*lo), Gf.Vec3f(*hi)]), t)
    stats.append({"date": date, "points_raw": int(len(d)), "points_usd": int(len(xyz)),
                  "height_m": round(float(xyz[:, 2].max() - xyz[:, 2].min()), 3)})

stage.SetStartTimeCode(0)
stage.SetEndTimeCode(len(days) - 1)
layer.customLayerData = {"generator": "pheno4d_replay (Pheno4D, Uni Bonn/ETH, PLoS ONE 2021)",
                         "plant": "Maize01", "unitScale": str(SCALE),
                         "scans": str(len(days))}
layer.Save()

print(f"{'day':>3} {'date':>6} {'raw pts':>9} {'usd pts':>8} {'height_m':>9}")
for i, s in enumerate(stats):
    print(f"{i:>3} {s['date']:>6} {s['points_raw']:>9} {s['points_usd']:>8} {s['height_m']:>9.3f}")
json.dump(stats, open("pheno_growth.json", "w"), indent=1)
if ON_KAGGLE:
    import shutil as sh
    for f in ("pheno_plant.usda", "pheno_growth.json"):
        sh.copy(f"{BASE}/{f}", os.path.join(WORK, f))
print("pheno_plant.usda persisted early:", round(os.path.getsize(f'{BASE}/pheno_plant.usda')/1e6, 1), "MB")


## 3. Compose into the farm twin stack (same architecture as FarmScan)
A `PHENO_LYR.usda` (asset layer, ASS_LYR's exact pattern) references the
plant; the copy of the root layer gains one sublayer entry. Then scrub the
composed stage across days - and plot the real growth curve.


In [ ]:
import glob as _glob, shutil, sys, tempfile
hits = _glob.glob("/kaggle/input/**/AGENTS.md", recursive=True) if ON_KAGGLE else []
TWIN_SRC = os.path.dirname(hits[0]) if hits else "../twin"
assert os.path.isfile(os.path.join(TWIN_SRC, "AGENTS.md")), "twin source not found"
print("twin source:", TWIN_SRC)

layers_dir = os.path.join(WORK, "twin_layers")
shutil.rmtree(layers_dir, ignore_errors=True)
shutil.copytree(os.path.join(TWIN_SRC, "assets/layers"), layers_dir)
shutil.copy(f"{BASE}/pheno_plant.usda", os.path.join(layers_dir, "pheno_plant.usda"))

pheno_lyr = '''#usda 1.0
(
    "PHENO layer - real scanned plant data (Pheno4D), time-sampled per scan day."
    defaultPrim = "World"
    metersPerUnit = 1
    upAxis = "Z"
)

def Xform "World"
{
    def Xform "Farm"
    {
        def Xform "PhenoPlant" (
            "Real maize plant, 14 daily laser scans as USD time samples (Pheno4D)."
            prepend references = @./pheno_plant.usda@
        )
        {
        }
    }
}
'''
open(os.path.join(layers_dir, "PHENO_LYR.usda"), "w").write(pheno_lyr)

sys.path.insert(0, os.path.join(TWIN_SRC, "exts/omniverse_crop_intelligence"))
from pxr import Sdf as _Sdf
root = _Sdf.Layer.FindOrOpen(os.path.join(layers_dir, "farm_twin.usda"))
if "./PHENO_LYR.usda" not in root.subLayerPaths:
    root.subLayerPaths.insert(0, "./PHENO_LYR.usda")   # root layer: only the sublayer list
    root.Save()
print("sublayers:", root.subLayerPaths)

from omniverse_crop_intelligence.stage_builder import StageBuilder
stack = StageBuilder(layers_dir).build(overwrite=False)
prim = stack.stage.GetPrimAtPath("/World/Farm/PhenoPlant/PlantCloud")
assert prim, "PhenoPlant did not compose"
cloud = UsdGeom.Points(prim)
for i, s in ((0, stats[0]), (len(stats)//2, stats[len(stats)//2]), (len(stats)-1, stats[-1])):
    pts_i = cloud.GetPointsAttr().Get(Usd.TimeCode(i))
    ext = cloud.GetExtentAttr().Get(Usd.TimeCode(i))
    print(f"day {i} (date {s['date']}): {len(pts_i)} points on the composed stage, height {ext[1][2]-ext[0][2]:.3f} m")
print("REAL PLANT GROWTH COMPOSED INTO THE TWIN: OK")


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
xs = list(range(len(stats)))
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(xs, [s["height_m"] for s in stats], "o-", color="#2e7d32", label="plant height [m]")
ax.set_xticks(xs)
ax.set_xticklabels([s["date"] for s in stats], rotation=45, fontsize=8)
ax.set_title("Pheno4D Maize01 - real daily scan growth (composed in the twin)")
ax.set_xlabel("scan date (2020)"); ax.set_ylabel("height [m]"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.savefig(f"{BASE}/pheno_growth.png", dpi=140)
if ON_KAGGLE:
    shutil.copy(f"{BASE}/pheno_growth.png", os.path.join(WORK, "pheno_growth.png"))
print("genuinely real growth curve saved")
for f in ("pheno_plant.usda", "pheno_growth.json", "pheno_growth.png"):
    p = os.path.join(WORK, f) if ON_KAGGLE else os.path.join(BASE, f)
    print(f"{f:>18}: {os.path.getsize(p)/1e6:8.2f} MB")
